# Multi Marginal Optimal Transport

In [6]:
import random
import torch
import numpy as np
import torch.nn as nn
from tqdm import tqdm
import torch.nn.functional as F
from torch.amp import GradScaler
from torch.utils.data import DataLoader
import deeplake
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score
from sklearn.decomposition import PCA
from geomloss import SamplesLoss
import time
import pandas as pd
from torch.utils.data import ConcatDataset, RandomSampler
from torch.optim.lr_scheduler import CosineAnnealingLR
import pickle
import seaborn as sns
import itertools
from dataset_OT import make_multi_WSI_dataset

seed = 42
torch.manual_seed(seed)
np.random.seed(42)
random.seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False 



In [7]:
#data loading


batch_size = 512 

print('Loading starting...')
idx_range_subset1 = [i for i in range(1,52+1)]
random.shuffle(idx_range_subset1)
num_train = int(np.ceil(0.7 * len(idx_range_subset1))) 
train_range1, val_range1 = idx_range_subset1[:num_train], idx_range_subset1[num_train:]

idx_range_subset3 = [i for i in range(1,26+1)] 
random.shuffle(idx_range_subset3)
num_train = int(np.ceil(0.7 * len(idx_range_subset3))) 
train_range3, val_range3 = idx_range_subset3[:num_train], idx_range_subset3[num_train:]

akoya_loader_train_subset1 = make_multi_WSI_dataset('Subset1', train_range1, ['Akoya'], train_or_test='Train', batch_size=batch_size)
akoya_loader_val_subset1 = make_multi_WSI_dataset('Subset1', val_range1, ['Akoya'], train_or_test='Train', batch_size=batch_size)
akoya_loader_train_subset3 = make_multi_WSI_dataset('Subset3', train_range3, ['Akoya'], train_or_test='Train', batch_size=batch_size)
akoya_loader_val_subset3 = make_multi_WSI_dataset('Subset3', val_range3, ['Akoya'], train_or_test='Train', batch_size=batch_size)
leica_loader_train = make_multi_WSI_dataset('Subset3', train_range3, ['Leica'], train_or_test='Train', batch_size=batch_size)
leica_loader_val = make_multi_WSI_dataset('Subset3', val_range3, ['Leica'], train_or_test='Train', batch_size=batch_size)
kfbio_loader_train = make_multi_WSI_dataset('Subset3', train_range3, ['KFBio'], train_or_test='Train', batch_size=batch_size)
kfbio_loader_val = make_multi_WSI_dataset('Subset3', val_range3, ['KFBio'], train_or_test='Train', batch_size=batch_size)

akoya_loader_train = ConcatDataset([akoya_loader_train_subset1, akoya_loader_train_subset3])
akoya_loader_val = ConcatDataset([akoya_loader_val_subset1, akoya_loader_val_subset3])

len_akoya_train = len(akoya_loader_train)
len_akoya_val = len(akoya_loader_val)

len_leica_train = len(leica_loader_train)
len_leica_val = len(leica_loader_val)

len_kfbio_train = len(kfbio_loader_train)
len_kfbio_val = len(kfbio_loader_val)

len_train = len_akoya_train + len_leica_train + len_kfbio_train
len_val = len_akoya_val + len_leica_val + len_kfbio_val

B_A_train = round(batch_size * len_akoya_train / (len_akoya_train + len_leica_train))
B_L_train = batch_size - B_A_train 
B_A_val = round(batch_size * len_akoya_val / (len_akoya_val + len_leica_val))
B_L_val = batch_size - B_A_val

akoya_loader_train = DataLoader(akoya_loader_train, batch_size=B_A_train, shuffle=True, num_workers=6, pin_memory=True, persistent_workers=True, prefetch_factor=4)
akoya_loader_val = DataLoader(akoya_loader_val, batch_size=B_A_val, num_workers=6, pin_memory=True, persistent_workers=True, prefetch_factor=4)
leica_loader_train = DataLoader(leica_loader_train, batch_size=B_L_train, shuffle=True, num_workers=6, pin_memory=True, persistent_workers=True, prefetch_factor=4)
leica_loader_val = DataLoader(leica_loader_val, batch_size=B_L_val, num_workers=6, pin_memory=True, persistent_workers=True, prefetch_factor=4)
kfbio_loader_train = DataLoader(kfbio_loader_train, batch_size=B_L_train, shuffle=True, num_workers=6, pin_memory=True, persistent_workers=True, prefetch_factor=4)
kfbio_loader_val = DataLoader(kfbio_loader_val, batch_size=B_L_val, num_workers=6, pin_memory=True, persistent_workers=True, prefetch_factor=4)

print("Train batches Akoya:", len(akoya_loader_train), 'batch size:', B_A_train)
print("Train batches Leica:", len(leica_loader_train), 'batch size:', B_L_train)
print("Train batches KFBio:", len(kfbio_loader_train), 'batch size:', B_L_train)
print("Validation batches Akoya:", len(akoya_loader_val), 'batch size:', B_A_val)
print("Validation batches Leica:", len(leica_loader_val), 'batch size:', B_L_val)

#in samples
print('len train:', len_train)

Loading starting...
Train batches Akoya: 4973 batch size: 399
Train batches Leica: 4978 batch size: 113
Train batches KFBio: 6062 batch size: 113
Validation batches Akoya: 1272 batch size: 388
Validation batches Leica: 1267 batch size: 124
len train: 3231403


In [8]:
class MultimarginalSinkhornDivergence(nn.Module):
    """
    Multimarginal Sinkhorn divergence for optimal transport between multiple domains.
    
    This implementation generalizes the Sinkhorn divergence to handle S > 2 marginals,
    suitable for penalizing models that produce different embeddings for different domains.
    """
    
    def __init__(self, 
                 epsilon: float = 0.01,
                 max_iter: int = 100,
                 threshold: float = 1e-3,
                 cost_fn: str = 'euclidean'):
        """
        Args:
            epsilon: Entropic regularization parameter
            max_iter: Maximum number of Sinkhorn iterations
            threshold: Convergence threshold
            cost_fn: Cost function type ('euclidean', 'cosine', 'manhattan')
        """
        super().__init__()
        self.epsilon = epsilon
        self.max_iter = max_iter
        self.threshold = threshold
        self.cost_fn = cost_fn
        
    def compute_cost_tensor(self, embeddings: List[torch.Tensor]) -> torch.Tensor:
        """
        Compute the multimarginal cost tensor C[i1, i2, ..., iS] between all embeddings.
        
        Args:
            embeddings: List of tensors with shapes [(n1, d), (n2, d), ..., (nS, d)]
            
        Returns:
            Cost tensor of shape (n1, n2, ..., nS)
        """
        S = len(embeddings)
        device = embeddings[0].device
        
        if self.cost_fn == 'euclidean':
            # For euclidean, we use sum of pairwise squared distances
            # C[i1,...,iS] = sum_{s<t} ||x_s^{i_s} - x_t^{i_t}||^2
            
            # Create meshgrids for all combinations
            indices = torch.meshgrid([torch.arange(emb.shape[0], device=device) 
                                    for emb in embeddings], indexing='ij')
            
            cost_shape = tuple(emb.shape[0] for emb in embeddings)
            cost_tensor = torch.zeros(cost_shape, device=device)
            
            # Sum over all pairs s < t
            for s in range(S):
                for t in range(s + 1, S):
                    # Get embeddings at corresponding indices
                    emb_s_selected = embeddings[s][indices[s]]  # Shape: cost_shape + (d,)
                    emb_t_selected = embeddings[t][indices[t]]  # Shape: cost_shape + (d,)
                    
                    # Compute squared L2 distance
                    pairwise_cost = torch.sum((emb_s_selected - emb_t_selected) ** 2, dim=-1)
                    cost_tensor += pairwise_cost
                    
        elif self.cost_fn == 'cosine':
            # Use cosine distance: 1 - cosine_similarity
            indices = torch.meshgrid([torch.arange(emb.shape[0], device=device) 
                                    for emb in embeddings], indexing='ij')
            
            cost_shape = tuple(emb.shape[0] for emb in embeddings)
            cost_tensor = torch.zeros(cost_shape, device=device)
            
            for s in range(S):
                for t in range(s + 1, S):
                    emb_s_selected = embeddings[s][indices[s]]
                    emb_t_selected = embeddings[t][indices[t]]
                    
                    # Normalize embeddings
                    emb_s_norm = F.normalize(emb_s_selected, p=2, dim=-1)
                    emb_t_norm = F.normalize(emb_t_selected, p=2, dim=-1)
                    
                    # Cosine similarity
                    cos_sim = torch.sum(emb_s_norm * emb_t_norm, dim=-1)
                    # Cosine distance
                    pairwise_cost = 1 - cos_sim
                    cost_tensor += pairwise_cost
                    
        elif self.cost_fn == 'manhattan':
            indices = torch.meshgrid([torch.arange(emb.shape[0], device=device) 
                                    for emb in embeddings], indexing='ij')
            
            cost_shape = tuple(emb.shape[0] for emb in embeddings)
            cost_tensor = torch.zeros(cost_shape, device=device)
            
            for s in range(S):
                for t in range(s + 1, S):
                    emb_s_selected = embeddings[s][indices[s]]
                    emb_t_selected = embeddings[t][indices[t]]
                    
                    pairwise_cost = torch.sum(torch.abs(emb_s_selected - emb_t_selected), dim=-1)
                    cost_tensor += pairwise_cost
        else:
            raise ValueError(f"Unknown cost function: {self.cost_fn}")
            
        return cost_tensor
    
    def sinkhorn_iterations(self, 
                          K: torch.Tensor, 
                          marginals: List[torch.Tensor]) -> Tuple[torch.Tensor, List[torch.Tensor]]:
        """
        Perform Sinkhorn iterations for multimarginal optimal transport.
        
        Args:
            K: Kernel tensor K = exp(-C/epsilon) of shape (n1, n2, ..., nS)
            marginals: List of marginal distributions [a1, a2, ..., aS]
            
        Returns:
            Optimal transport plan P and scaling vectors u
        """
        S = len(marginals)
        device = K.device
        
        # Initialize scaling vectors
        u_vectors = []
        for s in range(S):
            n_s = marginals[s].shape[0]
            u_vectors.append(torch.ones(n_s, device=device))
        
        # Sinkhorn iterations
        for iteration in range(self.max_iter):
            u_old = [u.clone() for u in u_vectors]
            
            # Update each scaling vector
            for s in range(S):
                # Compute denominator: sum over all indices except s
                # This is equivalent to marginalizing out all dimensions except s
                
                # Create tensor of scaling factors for all other dimensions
                scaling_tensor = K.clone()
                for r in range(S):
                    if r != s:
                        # Multiply by u_r along dimension r
                        shape = [1] * S
                        shape[r] = -1
                        u_r_reshaped = u_vectors[r].view(shape)
                        scaling_tensor = scaling_tensor * u_r_reshaped
                
                # Sum over all dimensions except s to get marginal
                sum_dims = list(range(S))
                sum_dims.remove(s)
                denominator = torch.sum(scaling_tensor, dim=sum_dims)
                
                # Update scaling vector
                u_vectors[s] = marginals[s] / (denominator + 1e-8)
            
            # Check convergence
            max_change = max(torch.max(torch.abs(u_vectors[s] - u_old[s])) 
                           for s in range(S))
            if max_change < self.threshold:
                break
        
        # Compute optimal transport plan
        P = K.clone()
        for s in range(S):
            shape = [1] * S
            shape[s] = -1
            u_s_reshaped = u_vectors[s].view(shape)
            P = P * u_s_reshaped
            
        return P, u_vectors
    
    def compute_sinkhorn_divergence(self, embeddings: List[torch.Tensor]) -> torch.Tensor:
        """
        Compute the Sinkhorn divergence between multiple embeddings.
        
        Args:
            embeddings: List of embedding tensors with different batch sizes but same feature dim
            
        Returns:
            Sinkhorn divergence value (scalar)
        """
        S = len(embeddings)
        device = embeddings[0].device
        
        # Create uniform marginals for each domain
        marginals = []
        for emb in embeddings:
            n = emb.shape[0]
            marginal = torch.ones(n, device=device) / n
            marginals.append(marginal)
        
        # Compute cost tensor
        C = self.compute_cost_tensor(embeddings)
        
        # Compute kernel K = exp(-C/epsilon)
        K = torch.exp(-C / self.epsilon)
        
        # Perform Sinkhorn iterations
        P, u_vectors = self.sinkhorn_iterations(K, marginals)
        
        # Compute primal objective: <P, C>
        primal_obj = torch.sum(P * C)
        
        # Compute entropy term: -epsilon * H(P)
        entropy_term = -self.epsilon * torch.sum(P * torch.log(P + 1e-8))
        
        # Sinkhorn divergence is the regularized optimal transport cost
        sinkhorn_div = primal_obj + entropy_term
        
        return sinkhorn_div
    
    def forward(self, embeddings: List[torch.Tensor]) -> torch.Tensor:
        """
        Forward pass computing Sinkhorn divergence.
        
        Args:
            embeddings: List of embedding tensors from different domains
            
        Returns:
            Sinkhorn divergence loss
        """
        return self.compute_sinkhorn_divergence(embeddings)


def multimarginal_sinkhorn_loss(embeddings: List[torch.Tensor],
                               epsilon: float = 0.01,
                               max_iter: int = 100,
                               cost_fn: str = 'euclidean') -> torch.Tensor:
    """
    Convenience function to compute multimarginal Sinkhorn divergence.
    
    Args:
        embeddings: List of embedding tensors
        epsilon: Regularization parameter
        max_iter: Maximum Sinkhorn iterations
        cost_fn: Cost function type
        
    Returns:
        Sinkhorn divergence loss
    """
    sinkhorn = MultimarginalSinkhornDivergence(
        epsilon=epsilon, 
        max_iter=max_iter, 
        cost_fn=cost_fn
    )
    return sinkhorn(embeddings)


# Example usage with your data structure:
def compute_ot_loss_for_batch(embedding_akoya, embedding_leica, embedding_kfbio,
                             epsilon=0.01, max_iter=100, cost_fn='euclidean'):
    """
    Compute OT loss for a single batch of embeddings from three domains.
    
    Args:
        embedding_akoya: Tensor of shape (batch_size_akoya, 1024)
        embedding_leica: Tensor of shape (batch_size_leica, 1024) 
        embedding_kfbio: Tensor of shape (batch_size_kfbio, 1024)
        
    Returns:
        Scalar loss value
    """
    embeddings = [embedding_akoya, embedding_leica, embedding_kfbio]
    return multimarginal_sinkhorn_loss(embeddings, epsilon, max_iter, cost_fn)


NameError: name 'List' is not defined

In [ ]:
# Test

sup_times = []
sinkhorn_divergence = MultimarginalSinkhornDivergence(epsilon=0.01, max_iter=50)

tq = tqdm(zip(akoya_loader_train, leica_loader_train, kfbio_loader_train),
                                                    desc=f"Timing OT losses",
                                                    total=min(len(akoya_loader_train), len(leica_loader_train), len(kfbio_loader_train)))

for batch_akoya, batch_leica, batch_kfbio in tq:
    embedding_akoya = batch_akoya['embedding']
    embedding_leica = batch_leica['embedding']
    embedding_kfbio = batch_kfbio['embedding']
    
    # Compute Sinkhorn divergence as loss
    t0 = time.time()
    ot_loss = sinkhorn_divergence([embedding_akoya, embedding_leica, embedding_kfbio])
    t1 = time.time()

    sup_times.append(t1 - t0)
    avg_sup = sum(sup_times) / len(sup_times)
    
    tq.set_description(f"Timing OT losses | Avg sup: {avg_sup:.3f}s | MMOT: {ot_loss}")
    break
    
        